# 04 — Gold KPIs

Builds the business-facing aggregate tables that Power BI will connect to directly.
Reads from the Silver model (`fact_order_items` + dimensions) and writes
pre-aggregated Gold tables to `delta/gold/`.

Note: "revenue" throughout this notebook means item `price` only (not freight/shipping),
since freight is a pass-through cost rather than product revenue.

In [3]:
from spark_utils import get_spark_session, stop_spark_session

# Spark's built-in column functions
from pyspark.sql import functions as F

# Window functions, needed for the RFM quartile scoring later in this notebook
from pyspark.sql.window import Window

# Start a Spark session tagged for this notebook's job
spark = get_spark_session("GoldKPIs")

# Where the Silver model tables live (written by 03_silver_linking_model.ipynb)
SILVER_DIR = "../delta/silver"

# Where this notebook will write the Gold (KPI) tables
GOLD_DIR = "../delta/gold"

D:\Udemy_Python\envs\pyspark-env\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [4]:
# Read the Silver model tables we'll need

fact_order_items = spark.read.format("delta").load(f"{SILVER_DIR}/fact_order_items")
dim_date = spark.read.format("delta").load(f"{SILVER_DIR}/dim_date")
dim_products = spark.read.format("delta").load(f"{SILVER_DIR}/dim_products")
dim_customers = spark.read.format("delta").load(f"{SILVER_DIR}/dim_customers")
dim_sellers = spark.read.format("delta").load(f"{SILVER_DIR}/dim_sellers")

print("Silver model tables loaded.")

Silver model tables loaded.


In [5]:
# --- gold_monthly_revenue ---

# Revenue, order count, and average order value by year/month.

gold_monthly_revenue = (
    fact_order_items
    .withColumn("purchase_date", F.to_date("order_purchase_timestamp"))   # strip time-of-day, keep just the date, to join against dim_date
    .join(dim_date, F.col("purchase_date") == dim_date["full_date"], how="left")  # bring in year/month columns
    .groupBy("year", "month", "month_name")
    .agg(
        F.sum("price").alias("total_revenue"),                     # sum of item prices (revenue, excluding freight)
        F.countDistinct("order_id").alias("order_count"),          # number of distinct orders in this month
    )
    .withColumn(
        "avg_order_value",
        F.col("total_revenue") / F.col("order_count")              # revenue per order for this month
    )
    .orderBy("year", "month")
)

gold_monthly_revenue.write.format("delta").mode("overwrite").save(f"{GOLD_DIR}/gold_monthly_revenue")

print(f"[Gold] monthly_revenue: {gold_monthly_revenue.count()} rows written")
gold_monthly_revenue.show(5)

[Gold] monthly_revenue: 24 rows written
+----+-----+----------+------------------+-----------+------------------+
|year|month|month_name|     total_revenue|order_count|   avg_order_value|
+----+-----+----------+------------------+-----------+------------------+
|2016|    9| September|            267.36|          3|             89.12|
|2016|   10|   October| 49507.65999999997|        308|160.73915584415573|
|2016|   12|  December|              10.9|          1|              10.9|
|2017|    1|   January|120312.87000000011|        789|152.48779467680623|
|2017|    2|  February| 247303.0200000006|       1733|142.70226197345679|
+----+-----+----------+------------------+-----------+------------------+
only showing top 5 rows


In [6]:
# --- gold_revenue_by_category ---

gold_revenue_by_category = (
    fact_order_items
    .join(dim_products, on="product_id", how="left")            # bring in the English category name
    .groupBy("product_category_name_english")
    .agg(
        F.sum("price").alias("total_revenue"),                  # revenue for this category
        F.countDistinct("order_id").alias("order_count"),       # distinct orders containing this category
    )
    .orderBy(F.col("total_revenue").desc())                     # highest-revenue categories first
)

gold_revenue_by_category.write.format("delta").mode("overwrite").save(f"{GOLD_DIR}/gold_revenue_by_category")

print(f"[Gold] revenue_by_category: {gold_revenue_by_category.count()} rows written")
gold_revenue_by_category.show(5)

[Gold] revenue_by_category: 72 rows written
+-----------------------------+------------------+-----------+
|product_category_name_english|     total_revenue|order_count|
+-----------------------------+------------------+-----------+
|                health_beauty|1258681.3399999887|       8836|
|                watches_gifts|1205005.6799999964|       5624|
|               bed_bath_table|1036988.6799999886|       9417|
|               sports_leisure| 988048.9699999907|       7720|
|         computers_accesso...|  911954.319999993|       6689|
+-----------------------------+------------------+-----------+
only showing top 5 rows


In [7]:
# --- gold_revenue_by_region ---

# fact_order_items already carries customer_id (joined in from orders back in Phase 3), so we can join straight to dim_customers here.

gold_revenue_by_region = (
    fact_order_items
    .join(dim_customers, on="customer_id", how="left")          # bring in customer_state
    .groupBy("customer_state")
    .agg(
        F.sum("price").alias("total_revenue"),                  # revenue attributed to this state
        F.countDistinct("order_id").alias("order_count"),       # distinct orders from this state
    )
    .orderBy(F.col("total_revenue").desc())                     # highest-revenue states first
)

gold_revenue_by_region.write.format("delta").mode("overwrite").save(f"{GOLD_DIR}/gold_revenue_by_region")

print(f"[Gold] revenue_by_region: {gold_revenue_by_region.count()} rows written")
gold_revenue_by_region.show(5)

[Gold] revenue_by_region: 27 rows written
+--------------+------------------+-----------+
|customer_state|     total_revenue|order_count|
+--------------+------------------+-----------+
|            SP| 5202955.049999867|      41375|
|            RJ|1824092.6700000146|      12762|
|            MG| 1585308.030000008|      11544|
|            RS| 750304.0199999922|       5432|
|            PR| 683083.7599999947|       4998|
+--------------+------------------+-----------+
only showing top 5 rows


In [8]:
# --- gold_delivery_by_category ---

# Does delivery speed relate to review scores, broken down by category?
# Kept as its own table (rather than merged with revenue_by_category) since it aggregates review_score/delivery_days rather than revenue.

gold_delivery_by_category = (
    fact_order_items
    .join(dim_products, on="product_id", how="left")
    .groupBy("product_category_name_english")
    .agg(
        F.avg("delivery_days").alias("avg_delivery_days"),           # average delivery time for this category
        F.avg("review_score").alias("avg_review_score"),             # average review score for this category
        F.countDistinct("order_id").alias("order_count"),            # sample size backing the averages above
    )
    .orderBy(F.col("avg_delivery_days").desc())                      # slowest-delivering categories first
)

gold_delivery_by_category.write.format("delta").mode("overwrite").save(f"{GOLD_DIR}/gold_delivery_by_category")

print(f"[Gold] delivery_by_category: {gold_delivery_by_category.count()} rows written")
gold_delivery_by_category.show(5)

[Gold] delivery_by_category: 72 rows written
+-----------------------------+------------------+------------------+-----------+
|product_category_name_english| avg_delivery_days|  avg_review_score|order_count|
+-----------------------------+------------------+------------------+-----------+
|             office_furniture|20.786570743405274|  3.48896839594514|       1273|
|           christmas_supplies|15.673333333333334|  4.02054794520548|        128|
|                fashion_shoes|15.396887159533074| 4.238372093023256|        240|
|         security_and_serv...|              15.0|               2.5|          2|
|               home_comfort_2|14.533333333333333|3.6296296296296298|         24|
+-----------------------------+------------------+------------------+-----------+
only showing top 5 rows


In [9]:
# --- gold_delivery_by_region ---

# Same idea as above, broken down by customer state instead of category.

gold_delivery_by_region = (
    fact_order_items
    .join(dim_customers, on="customer_id", how="left")
    .groupBy("customer_state")
    .agg(
        F.avg("delivery_days").alias("avg_delivery_days"),
        F.avg("review_score").alias("avg_review_score"),
        F.countDistinct("order_id").alias("order_count"),
    )
    .orderBy(F.col("avg_delivery_days").desc())
)

gold_delivery_by_region.write.format("delta").mode("overwrite").save(f"{GOLD_DIR}/gold_delivery_by_region")

print(f"[Gold] delivery_by_region: {gold_delivery_by_region.count()} rows written")
gold_delivery_by_region.show(5)

[Gold] delivery_by_region: 27 rows written
+--------------+------------------+------------------+-----------+
|customer_state| avg_delivery_days|  avg_review_score|order_count|
+--------------+------------------+------------------+-----------+
|            AP| 28.22222222222222| 4.222222222222222|         68|
|            RR| 28.17391304347826| 3.576923076923077|         46|
|            AM|26.337423312883434|4.0920245398773005|        147|
|            AL|24.447306791569087|  3.72562358276644|        411|
|            PA|23.702087286527515|3.7877358490566038|        970|
+--------------+------------------+------------------+-----------+
only showing top 5 rows


In [10]:
# --- gold_seller_performance ---

gold_seller_performance = (
    fact_order_items
    .join(dim_sellers, on="seller_id", how="left")               # bring in seller_state
    .groupBy("seller_id", "seller_state")
    .agg(
        F.sum("price").alias("total_revenue"),                   # revenue generated by this seller
        F.countDistinct("order_id").alias("order_count"),        # distinct orders this seller fulfilled
        F.avg("review_score").alias("avg_review_score"),         # average review score across their orders
    )
    .orderBy(F.col("total_revenue").desc())                      # top sellers by revenue first
)

gold_seller_performance.write.format("delta").mode("overwrite").save(f"{GOLD_DIR}/gold_seller_performance")

print(f"[Gold] seller_performance: {gold_seller_performance.count()} rows written")
gold_seller_performance.show(5)

[Gold] seller_performance: 3095 rows written
+--------------------+------------+------------------+-----------+------------------+
|           seller_id|seller_state|     total_revenue|order_count|  avg_review_score|
+--------------------+------------+------------------+-----------+------------------+
|4869f7a5dfa277a7d...|          SP|229472.63000000027|       1132| 4.122822299651568|
|53243585a1d6dc264...|          BA|222776.05000000005|        358| 4.075980392156863|
|4a3ca9315b744ce9f...|          SP|200472.92000000033|       1806| 3.803261977573904|
|fa1c13f2614d7b5c4...|          SP| 194042.0300000001|        585|  4.34020618556701|
|7c67e1448b00f6e96...|          SP| 187923.8900000001|        982|3.3414454277286136|
+--------------------+------------+------------------+-----------+------------------+
only showing top 5 rows


In [11]:
# --- gold_rfm_segments: Step 1 — one row per order (not per line item) ---
# total_payment_value is repeated across every line item of the same order,
# so summing it straight off fact_order_items would overcount. Deduplicate
# down to order grain first.

orders_deduped = (
    fact_order_items
    .select("order_id", "customer_id", "order_purchase_timestamp", "total_payment_value")
    .dropDuplicates(["order_id"])                                # one row per order_id, dropping the extra line-item rows
)

print(f"orders_deduped: {orders_deduped.count()} rows (should equal distinct order_id count)")

orders_deduped: 98666 rows (should equal distinct order_id count)


In [12]:
# --- gold_rfm_segments: Step 2 — link to the real person (customer_unique_id) ---
# Olist generates a fresh customer_id per order; customer_unique_id is what
# actually identifies the same person across multiple orders.

orders_with_person = orders_deduped.join(
    dim_customers.select("customer_id", "customer_unique_id"),    # only need these two columns from dim_customers
    on="customer_id",
    how="left"
)

print(f"orders_with_person: {orders_with_person.count()} rows")

orders_with_person: 98666 rows


In [13]:
# --- gold_rfm_segments: Step 3 — compute raw Recency, Frequency, Monetary per person ---

# Reference date = the single latest order date across the whole dataset (this is historical 2016-2018 data, so "today" wouldn't make sense here)
reference_date = orders_with_person.agg(F.max("order_purchase_timestamp")).collect()[0][0]
print(f"RFM reference date: {reference_date}")

rfm_raw = (
    orders_with_person
    .groupBy("customer_unique_id")
    .agg(
        F.max("order_purchase_timestamp").alias("last_order_date"),   # this person's most recent order
        F.countDistinct("order_id").alias("frequency"),               # how many distinct orders this person placed
        F.sum("total_payment_value").alias("monetary"),               # total amount this person has spent
    )
    .withColumn(
        "recency_days",
        F.datediff(F.lit(reference_date), F.col("last_order_date"))   # days between reference date and their last order
    )
)

rfm_raw.show(5)

RFM reference date: 2018-09-03 09:06:57
+--------------------+-------------------+---------+--------+------------+
|  customer_unique_id|    last_order_date|frequency|monetary|recency_days|
+--------------------+-------------------+---------+--------+------------+
|861eff4711a542e4b...|2017-05-16 15:05:35|        1|  146.87|         475|
|9dfe23c51c1758f19...|2017-11-29 21:40:21|        1|   57.55|         278|
|14e7273c0d9ae09dd...|2018-04-03 11:05:02|        1|  301.42|         153|
|85b3ae9df63745fe6...|2017-08-01 13:43:33|        1|   39.78|         398|
|8777621c1d67b4d69...|2017-03-23 22:36:21|        1|  154.29|         529|
+--------------------+-------------------+---------+--------+------------+
only showing top 5 rows


In [14]:
# --- gold_rfm_segments: Step 4 — score each metric into quartiles 1-4 ---
# ntile(4) splits the population into 4 equal-sized buckets based on order.
# We want "4 = best" consistently across all three metrics:
#   - Recency: FEWER days since last order is better, so order descending (largest recency_days first) - that puts the most-recent customers 
#     in the last bucket, which ntile labels 4.
#   - Frequency & Monetary: HIGHER is better, so we order ascending as normal - the highest values land in the last bucket, labeled 4.

recency_window = Window.orderBy(F.col("recency_days").desc())    # largest (worst) recency first -> smallest (best) ends up in bucket 4
frequency_window = Window.orderBy(F.col("frequency").asc())      # smallest (worst) frequency first -> largest ends up in bucket 4
monetary_window = Window.orderBy(F.col("monetary").asc())        # smallest (worst) monetary first -> largest ends up in bucket 4

rfm_scored = (
    rfm_raw
    .withColumn("r_score", F.ntile(4).over(recency_window))      # 1 (long time since ordering) to 4 (ordered very recently)
    .withColumn("f_score", F.ntile(4).over(frequency_window))    # 1 (rarely orders) to 4 (orders often)
    .withColumn("m_score", F.ntile(4).over(monetary_window))     # 1 (low spend) to 4 (high spend)
)

rfm_scored.select("customer_unique_id", "recency_days", "frequency", "monetary", "r_score", "f_score", "m_score").show(5)

+--------------------+------------+---------+--------+-------+-------+-------+
|  customer_unique_id|recency_days|frequency|monetary|r_score|f_score|m_score|
+--------------------+------------+---------+--------+-------+-------+-------+
|830d5b7aaa3b6f1e9...|         718|        1|    NULL|      1|      1|      1|
|317cfc692e3f86c45...|           9|        1|    9.59|      4|      4|      1|
|bd06ce0e06ad77a7f...|         355|        1|   10.07|      1|      1|      1|
|b33336f46234b24a6...|          74|        1|   10.89|      4|      4|      1|
|6f5b9d1cdccc4d28f...|         366|        1|   11.63|      1|      1|      1|
+--------------------+------------+---------+--------+-------+-------+-------+
only showing top 5 rows


In [15]:
# --- gold_rfm_segments: Step 5 — combine scores into named segments ---
# A simple, explainable rule table rather than a clustering model -
# easier to explain on a dashboard.

gold_rfm_segments = rfm_scored.withColumn(
    "segment",
    F.when((F.col("r_score") >= 4) & (F.col("f_score") >= 4) & (F.col("m_score") >= 4), "Champions")
     .when((F.col("f_score") >= 3) & (F.col("m_score") >= 3), "Loyal Customers")
     .when((F.col("r_score") <= 2) & ((F.col("f_score") >= 3) | (F.col("m_score") >= 3)), "At Risk")
     .when((F.col("r_score") <= 2) & (F.col("f_score") <= 2) & (F.col("m_score") <= 2), "Lost")
     .otherwise("New / Low Engagement")                                    # everyone not caught by the rules above
)

gold_rfm_segments.write.format("delta").mode("overwrite").save(f"{GOLD_DIR}/gold_rfm_segments")

print(f"[Gold] rfm_segments: {gold_rfm_segments.count()} rows written")

# Show how many customers fell into each segment
gold_rfm_segments.groupBy("segment").count().orderBy(F.col("count").desc()).show()

[Gold] rfm_segments: 95420 rows written
+--------------------+-----+
|             segment|count|
+--------------------+-----+
|                Lost|24179|
|New / Low Engagement|23987|
|             At Risk|22392|
|     Loyal Customers|19294|
|           Champions| 5568|
+--------------------+-----+



In [16]:
stop_spark_session(spark)